# updatesupport downstream of EconML

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nahuaque/updatesupport/blob/main/examples/notebooks/econml_downstream_reporting_colab.ipynb)

This notebook shows how `updatesupport` sits downstream of [EconML](https://www.pywhy.org/EconML/).

The workflow is:

1. Fit an EconML conditional-treatment-effect estimator.
2. Compute `tau_hat = estimator.effect(X)` for each row.
3. Ask whether a coarse public report can stably summarize those estimated effects.

`updatesupport` is not replacing the causal estimator. EconML handles the first-stage causal estimation. `updatesupport` audits the reporting layer: if we publish effects by coarse public buckets, how much could the aggregate reported effect move if hidden composition inside those buckets changed?

## 1. Install and import

Run the install cell once in Colab. It pins a compatible EconML / NumPy / scikit-learn stack plus the causal and plotting extras used in this tutorial.

If you previously imported NumPy, SciPy, scikit-learn, or EconML in the same runtime, restart the runtime after this install cell before continuing.

In [ ]:
%pip install -q --upgrade --force-reinstall --no-cache-dir "updatesupport[causal,examples]>=0.1.2" "econml==0.16.0" "numpy==2.2.6" "scipy==1.15.3" "scikit-learn==1.6.1" "pandas>=2.2,<2.4" "seaborn>=0.13,<0.14" "matplotlib>=3.9,<3.11" "ipywidgets>=8,<9"

try:
    from google.colab import output

    output.enable_custom_widget_manager()
except Exception:
    pass

In [ ]:
from importlib.metadata import version

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from econml.dml import CausalForestDML
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

import updatesupport as us

print(f"updatesupport {version('updatesupport')}")
print(f"econml {version('econml')}")
print(f"numpy {version('numpy')}")
print(f"scikit-learn {version('scikit-learn')}")

sns.set_theme(
    style="whitegrid",
    context="notebook",
    palette="colorblind",
    rc={"axes.spines.right": False, "axes.spines.top": False},
)

## 2. Build a synthetic causal dataset

The synthetic data has a known heterogeneous treatment effect, `true_tau`, but the audit below will use EconML's estimated effect, `tau_hat`.

The public report will only use `age_band`, `region`, and `sex`. The hidden state also includes acquisition channel, prior-use behavior, risk band, and severity band. That mimics a common model-review situation: the public report is simpler than the internal feature space.

In [ ]:
RNG = np.random.default_rng(42)
N = 900

age = RNG.normal(44, 13, N).clip(18, 78)
income = RNG.lognormal(mean=10.45, sigma=0.45, size=N)
severity = RNG.beta(2.2, 4.0, N)
prior_use = RNG.poisson(1.4 + 2.5 * severity, N)
risk_score = (0.55 * severity + 0.015 * prior_use + RNG.normal(0, 0.08, N)).clip(0, 1)

region = RNG.choice(["north", "south", "west"], size=N, p=[0.36, 0.34, 0.30])
sex = RNG.choice(["F", "M"], size=N, p=[0.52, 0.48])
channel = RNG.choice(["direct", "partner", "online"], size=N, p=[0.40, 0.28, 0.32])

age_band = pd.cut(
    age,
    bins=[17, 34, 49, 64, 90],
    labels=["18_34", "35_49", "50_64", "65_plus"],
).astype(str)
risk_band = pd.cut(
    risk_score,
    bins=[-0.01, 0.25, 0.45, 1.01],
    labels=["low", "medium", "high"],
).astype(str)
severity_band = pd.cut(
    severity,
    bins=[-0.01, 0.25, 0.50, 1.01],
    labels=["low", "medium", "high"],
).astype(str)
prior_use_band = pd.cut(
    prior_use,
    bins=[-1, 1, 4, 99],
    labels=["0_1", "2_4", "5_plus"],
).astype(str)

region_effect = (
    pd.Series(region).map({"north": 0.04, "south": -0.02, "west": 0.01}).to_numpy()
)
channel_effect = (
    pd.Series(channel)
    .map({"direct": 0.02, "partner": -0.03, "online": 0.05})
    .to_numpy()
)
true_tau = 0.18 + 0.20 * severity + 0.08 * risk_score + region_effect + channel_effect

propensity_logit = (
    -0.35
    + 1.1 * severity
    + 0.40 * (region == "south")
    + 0.35 * (channel == "online")
    - 0.25 * (sex == "F")
)
propensity = 1 / (1 + np.exp(-propensity_logit))
treatment = RNG.binomial(1, propensity)

baseline = (
    0.30
    + 0.006 * (age - 44)
    + 0.0000018 * (income - income.mean())
    + 0.35 * severity
    + 0.08 * (region == "south")
    - 0.04 * (channel == "direct")
)
outcome = baseline + true_tau * treatment + RNG.normal(0, 0.18, N)
sample_weight = RNG.uniform(0.7, 1.6, N)

df = pd.DataFrame(
    {
        "age": age,
        "income": income,
        "severity": severity,
        "prior_use": prior_use,
        "risk_score": risk_score,
        "region": region,
        "sex": sex,
        "channel": channel,
        "age_band": age_band,
        "risk_band": risk_band,
        "severity_band": severity_band,
        "prior_use_band": prior_use_band,
        "treatment": treatment,
        "outcome": outcome,
        "true_tau": true_tau,
        "sample_weight": sample_weight,
    }
)

PUBLIC_COLUMNS = ("age_band", "region", "sex")
HIDDEN_COLUMNS = (
    "age_band",
    "region",
    "sex",
    "channel",
    "risk_band",
    "severity_band",
    "prior_use_band",
)
CANDIDATE_REFINEMENTS = ("channel", "risk_band", "severity_band", "prior_use_band")

df.head()

## 3. Fit EconML and attach `tau_hat`

This is the upstream causal step. We fit `CausalForestDML`, then call `estimator.effect(X)`. `updatesupport.adapt_econml_effects(...)` copies those effect estimates into the rows as `tau_hat` and records lightweight adapter metadata.

In a real analysis, identification assumptions, nuisance models, overlap checks, and causal diagnostics live here. The support audit starts after the effect estimates exist.

In [ ]:
X_COLUMNS = ["age", "income", "severity", "prior_use", "risk_score"]
W_COLUMNS = ["region", "sex", "channel"]

X = df.loc[:, X_COLUMNS]
W = pd.get_dummies(df.loc[:, W_COLUMNS], drop_first=True)
Y = df["outcome"].to_numpy()
T = df["treatment"].to_numpy()

estimator = CausalForestDML(
    model_y=RandomForestRegressor(
        n_estimators=80,
        min_samples_leaf=20,
        random_state=7,
    ),
    model_t=RandomForestClassifier(
        n_estimators=80,
        min_samples_leaf=20,
        random_state=8,
    ),
    discrete_treatment=True,
    n_estimators=96,
    min_samples_leaf=15,
    max_depth=8,
    inference=False,
    random_state=9,
)
estimator.fit(Y, T, X=X, W=W, sample_weight=df["sample_weight"].to_numpy())

adapted = us.adapt_econml_effects(
    estimator,
    df,
    X,
    effect_column="tau_hat",
)
effect_df = pd.DataFrame(adapted.rows)

print(f"adapter source: {adapted.source}")
print(f"estimator: {adapted.estimator_name}")
print(f"effect column: {adapted.effect_column}")
effect_df[
    ["true_tau", "tau_hat", "age_band", "region", "sex", "channel", "risk_band"]
].head()

## 4. Inspect estimated effects before auditing support

The left chart checks whether the estimated effects track the synthetic ground truth. The right chart shows how `tau_hat` varies across hidden cells. That hidden variation is what can make a coarse public report unstable.

In [ ]:
hidden_summary = (
    effect_df.groupby(
        ["age_band", "region", "sex", "channel", "risk_band"], as_index=False
    )
    .agg(
        rows=("tau_hat", "size"),
        sample_weight=("sample_weight", "sum"),
        tau_hat=("tau_hat", "mean"),
        true_tau=("true_tau", "mean"),
    )
    .query("rows >= 3")
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)
sns.scatterplot(
    data=effect_df.sample(350, random_state=1),
    x="true_tau",
    y="tau_hat",
    hue="risk_band",
    alpha=0.75,
    ax=axes[0],
)
axes[0].plot(
    [effect_df["true_tau"].min(), effect_df["true_tau"].max()],
    [effect_df["true_tau"].min(), effect_df["true_tau"].max()],
    color="#C44E52",
    linestyle="--",
    linewidth=1.5,
)
axes[0].set_title("EconML estimated CATE vs synthetic truth")
axes[0].set_xlabel("true_tau")
axes[0].set_ylabel("tau_hat")

sns.boxplot(data=effect_df, x="risk_band", y="tau_hat", hue="channel", ax=axes[1])
axes[1].set_title("Estimated effect heterogeneity in hidden fields")
axes[1].set_xlabel("Risk band")
axes[1].set_ylabel("tau_hat")
plt.show()

## 5. Audit the public causal report

Now the causal estimator is finished. The audit asks a reporting question:

> If we publish effects only by `age_band x region x sex`, could the aggregate reported `tau_hat` change if the hidden mix of `channel`, `risk_band`, `severity_band`, and `prior_use_band` changed inside those public buckets?

The hidden-composition interval is not a confidence interval. It is a support-stability interval under the chosen hidden-mix stress test.

In [ ]:
def build_effect_report(q_radius: float = 0.45, ambiguity_limit: float = 0.015):
    return adapted.audit_effects(
        public=PUBLIC_COLUMNS,
        hidden=HIDDEN_COLUMNS,
        weight="sample_weight",
        candidate_refinements=CANDIDATE_REFINEMENTS,
        q=us.q_bounded_shift(q_radius),
        min_cell_weight=3,
        title="EconML Estimated-Effect Reporting Stability Audit",
        top=8,
    )


def report_tables(report):
    return {
        name: pd.DataFrame(rows) for name, rows in report.to_tables().items() if rows
    }


@widgets.interact(
    q_radius=widgets.FloatSlider(
        value=0.45, min=0.0, max=0.90, step=0.05, description="Q radius"
    ),
    ambiguity_limit=widgets.FloatSlider(
        value=0.015,
        min=0.0,
        max=0.060,
        step=0.0025,
        readout_format=".3f",
        description="Limit",
    ),
)
def audit_dashboard(q_radius: float, ambiguity_limit: float):
    report = build_effect_report(q_radius=q_radius, ambiguity_limit=ambiguity_limit)
    tables = report_tables(report)
    summary = tables["summary"].iloc[0]
    fibers = tables.get("worst_fibers", pd.DataFrame())
    refinements = tables.get("refinements", pd.DataFrame())

    print(f"observed aggregate tau_hat: {summary['observed_value']:.4f}")
    print(
        f"hidden-composition interval: [{summary['lower']:.4f}, {summary['upper']:.4f}]"
    )
    print(f"ambiguity width: {summary['ambiguity']:.4f}")
    print(f"public adequate: {summary['public_adequate']}")

    fig, axes = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)
    if not fibers.empty:
        fibers = fibers.assign(
            public_label=fibers["public_value"].map(
                lambda value: " | ".join(map(str, value))
            )
        )
        sns.barplot(
            data=fibers.sort_values("contribution", ascending=False).head(8),
            y="public_label",
            x="contribution",
            color="#4C72B0",
            ax=axes[0],
        )
        axes[0].set_title("Public buckets driving ambiguity")
        axes[0].set_xlabel("Contribution to ambiguity")
        axes[0].set_ylabel("")
    if not refinements.empty:
        sns.barplot(
            data=refinements.sort_values("reduction", ascending=False),
            y="column",
            x="reduction",
            color="#55A868",
            ax=axes[1],
        )
        axes[1].set_title("Best hidden-field refinements")
        axes[1].set_xlabel("Ambiguity reduction")
        axes[1].set_ylabel("")
    plt.show()

## 6. Read the report artifact

The Markdown report is the attachable artifact. It separates the estimated causal target from hidden-composition ambiguity and refinement recommendations. In a real model review, you would attach this downstream of the EconML estimation notebook or pipeline run.

In [ ]:
report = build_effect_report()
markdown = report.to_markdown()
print("\n".join(markdown.splitlines()[:80]))

## 7. Search for a smaller stable public representation

The frontier search asks which hidden fields are worth promoting into the public report. The goal is not to publish every internal covariate. The goal is to find a public representation that is small enough to communicate and stable enough for review.

In [ ]:
def build_frontier(q_radius: float = 0.45, ambiguity_limit: float = 0.015):
    return us.public_representation_frontier(
        adapted.rows,
        base_public=PUBLIC_COLUMNS,
        hidden=HIDDEN_COLUMNS,
        target="tau_hat",
        weight="sample_weight",
        candidate_refinements=CANDIDATE_REFINEMENTS,
        q_presets=("saturated", us.q_bounded_shift(q_radius), "observed"),
        min_cell_weights=(3,),
        ambiguity_limit=ambiguity_limit,
        bucket_budget=80,
        search="beam",
        beam_width=8,
        max_added_columns=3,
        max_evaluations=96,
        title="EconML Downstream Public Representation Frontier",
    )


@widgets.interact(
    q_radius=widgets.FloatSlider(
        value=0.45, min=0.0, max=0.90, step=0.05, description="Q radius"
    ),
    ambiguity_limit=widgets.FloatSlider(
        value=0.015,
        min=0.0,
        max=0.060,
        step=0.0025,
        readout_format=".3f",
        description="Limit",
    ),
)
def frontier_dashboard(q_radius: float, ambiguity_limit: float):
    frontier = build_frontier(q_radius=q_radius, ambiguity_limit=ambiguity_limit)
    rows = [
        {
            "label": candidate.label,
            "public_cells": candidate.public_cells,
            "max_ambiguity": candidate.max_ambiguity,
            "added_columns": candidate.added_column_count,
            "stable": bool(candidate.passes_ambiguity_limit),
            "frontier": candidate in frontier.frontier,
        }
        for candidate in frontier.candidates
    ]
    frame = pd.DataFrame(rows)
    fig, ax = plt.subplots(figsize=(11, 6))
    sns.scatterplot(
        data=frame,
        x="public_cells",
        y="max_ambiguity",
        hue="stable",
        style="frontier",
        size="added_columns",
        sizes=(80, 320),
        ax=ax,
    )
    ax.axhline(
        ambiguity_limit,
        color="#C44E52",
        linestyle="--",
        linewidth=1.5,
        label="review limit",
    )
    ax.set_title("Public representation frontier for tau_hat")
    ax.set_xlabel("Public cells")
    ax.set_ylabel("Worst ambiguity")
    plt.show()

    if frontier.minimal_stable is not None:
        print(f"minimal stable representation: {frontier.minimal_stable.label}")
    else:
        print("no evaluated representation meets the ambiguity limit")

## 8. Takeaway

The handoff is simple:

```python
estimator.fit(Y, T, X=X, W=W)
adapted = us.adapt_econml_effects(estimator, df, X)
report = adapted.audit_effects(...)
```

EconML estimates heterogeneous effects. `updatesupport` asks whether the public representation used to report those effects is stable under hidden-composition shifts. That makes the support audit complementary to, not competitive with, causal estimation.